# FACT ORDERS

### Data Reading

In [0]:
df = spark.sql('select * from project.silver.orders')
display(df.limit(10))

In [0]:
df_dim_cust = spark.sql('select dimKeyCustomer, customer_id as dimCustKey from project.gold.customers')

df_dim_pro = spark.sql('select product_id as dimKeyProduct, product_id  as dimProKey from project.gold.products')


In [0]:
df_fact = df.join(df_dim_cust,df['customer_id']==df_dim_cust['dimCustKey'], how='inner').join(df_dim_pro,df['product_id']==df_dim_pro['dimProKey'], how='inner')

In [0]:
df_fact = df_fact.drop('dimCustKey','dimProKey','customer_id','product_id')

In [0]:
df_fact.display()

### Upsert on Fact Table

In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("project.gold.orders"):
    dlt_obj = DeltaTable.forName(spark, 'project.gold.orders')
    dlt_obj.alias('trg').merge(df_fact.alias('src'), 'trg.order_id == src.order_id AND trg.dimKeyCustomer == src.dimKeyCustomer and trg.dimKeyProduct = src.dimKeyProduct')
else:
    df_fact.write.format("delta") \
        .saveAsTable('project.gold.orders')


In [0]:
%sql
select * from project.gold.orders